In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import ImageFile
import gc  # garbage collector to free memory

2025-10-31 15:04:31.047594: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-31 15:04:31.055148: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-31 15:04:31.110631: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-31 15:04:31.158365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761923071.206058   12363 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761923071.22

In [2]:
print("All modules imported successfully.")

All modules imported successfully.


In [3]:
# ==============================
# Step 1: Load dataset
# ==============================
data_dir = "../../data sheets/training_set"

In [4]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [5]:
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [6]:
batch_size = 8  # smaller batch to avoid memory overload
img_size = (224, 224)

In [7]:
generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",
    shuffle=False
)

Found 5000 images belonging to 5 classes.


In [8]:
# ==============================
# Step 2: Feature extraction (safe version)
# ==============================
base_model = VGG16(weights="imagenet", include_top=False, pooling="avg")

2025-10-31 15:04:39.943838: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2025-10-31 15:04:39.943880: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-10-31 15:04:39.943885: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: thathsara-bandara-LOQ-15IRX9
2025-10-31 15:04:39.943888: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:190] hostname: thathsara-bandara-LOQ-15IRX9
2025-10-31 15:04:39.944252: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:197] libcuda reported version is: NOT_FOUND: was unable to find libcuda.so DSO loaded into this program. The library may be missing or provided via another ob

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 58s 1us/step


In [9]:
print("Extracting features batch by batch...")

Extracting features batch by batch...


In [10]:
features_list = []
labels_list = []

In [11]:
num_batches = int(np.ceil(generator.n / batch_size))

In [12]:
for i in range(num_batches):
    x_batch, y_batch = next(generator)  # ✅ fixed line
    feat_batch = base_model.predict(x_batch, verbose=0)
    features_list.append(feat_batch)
    labels_list.append(y_batch)
    
    print(f"Processed batch {i+1}/{num_batches}")
    
    # Free memory
    del x_batch, y_batch, feat_batch
    gc.collect()

Processed batch 1/625
Processed batch 2/625
Processed batch 3/625
Processed batch 4/625
Processed batch 5/625
Processed batch 6/625
Processed batch 7/625
Processed batch 8/625
Processed batch 9/625
Processed batch 10/625
Processed batch 11/625
Processed batch 12/625
Processed batch 13/625
Processed batch 14/625
Processed batch 15/625
Processed batch 16/625
Processed batch 17/625
Processed batch 18/625
Processed batch 19/625
Processed batch 20/625
Processed batch 21/625
Processed batch 22/625
Processed batch 23/625
Processed batch 24/625
Processed batch 25/625
Processed batch 26/625
Processed batch 27/625
Processed batch 28/625
Processed batch 29/625
Processed batch 30/625
Processed batch 31/625
Processed batch 32/625
Processed batch 33/625
Processed batch 34/625
Processed batch 35/625
Processed batch 36/625
Processed batch 37/625
Processed batch 38/625
Processed batch 39/625
Processed batch 40/625
Processed batch 41/625
Processed batch 42/625
Processed batch 43/625
Processed batch 44/6

In [13]:
# Combine all batches into arrays
features = np.vstack(features_list)
labels = np.hstack(labels_list)

In [14]:
print("Feature extraction complete.")
print("Feature shape:", features.shape)
print("Labels shape", labels.shape)

Feature extraction complete.
Feature shape: (5000, 512)
Labels shape (5000,)


In [15]:
# ==============================
# Step 3: Train Random Forest classifier
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

In [16]:
clf = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1
)

In [17]:
print("Training Random Forest classifier...")
clf.fit(X_train, y_train)
print("Training complete.")

Training Random Forest classifier...
Training complete.


In [18]:
# ==============================
# Step 4: Evaluate model
# ==============================
y_pred = clf.predict(X_test)

In [19]:
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", round(accuracy * 100, 2), "%")


Accuracy: 39.9 %


In [20]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

         0.0       0.35      0.40      0.37       200
         1.0       0.43      0.47      0.45       200
         2.0       0.33      0.20      0.25       200
         3.0       0.46      0.47      0.46       200
         4.0       0.41      0.47      0.44       200

    accuracy                           0.40      1000
   macro avg       0.39      0.40      0.39      1000
weighted avg       0.39      0.40      0.39      1000



In [21]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[79 50 22 18 31]
 [48 94 12 19 27]
 [41 34 40 44 41]
 [29 16 27 93 35]
 [30 27 22 28 93]]


In [22]:
# ==============================
# Step 5: Predict face shape names
# ==============================
class_labels = {v: k for k, v in generator.class_indices.items()}

In [23]:
predicted_shapes = [class_labels[int(p)] for p in y_pred]
print("\nExample Predictions (first 10):")
print(predicted_shapes[:10])


Example Predictions (first 10):
['Round', 'Round', 'Heart', 'Oval', 'Oblong', 'Heart', 'Square', 'Oblong', 'Heart', 'Oval']
